# 실험 G: 함수 토큰 대신 JSON 으로 부르는 일반 모델 (논문에 없는 추가)
설계·예측·판정 규칙은 `EXPERIMENT_DESIGN.md` 4.5절 (돌리기 전에 적어 둠). 코드는 `src/general.py`, `src/train.py --format json`.

**묻는 것**: 지금까지 본 거절 현상(낯선 말투 positive 과잉 거절, near-miss 오호출, 문턱값 이식 실패, 말투 의존)이
함수 토큰 방식이라서 생기나, 함수 호출 모델이면 공통으로 생기나.

| | `ours` (기존, 다시 학습 안 함) | `G` (새로 학습) |
|---|---|---|
| 프롬프트 | 지시문 + 질의 | 지시문 + **함수 21개 JSON 스키마** + 질의 (약 2,350 토큰) |
| 정답 | ` <nexa_11>(5)<nexa_end>` + Function description | ` {"name": "change_screen_brightness", "arguments": {"level": 5}}` |
| 거절 | `<nexa_20>` | `{"name": "irrelevant_function", "arguments": {}}` (목록 21번째) |
| 새 토큰 | 22개 추가 | 없음 |
| 같은 것 | gemma-2b, `data/train.jsonl` 4,000개, full FT, lr 5e-5, 3 epoch, batch 16, seed 0 | 같음 |

**거절 점수**: G 는 함수 이름이 여러 토큰이라, 이름 21개를 하나씩 정답 자리에 넣어 그 이름을 쓸 확률을 계산하고 21개로 나눈다.
`probs.jsonl` 형식이 기존과 같아서 `refusal_metrics`·`negcat_report`·`style_report` 를 그대로 쓴다.
단 ours 의 확률은 전체 사전 기준, G 는 21개 안에서 다시 나눈 값이라 **문턱값 숫자는 두 모델끼리 직접 비교하지 않는다** (AUROC·거절 개수로 비교).

**런타임**: A100 80GB. 문장이 ours 의 약 21배 길어서 학습이 수 시간 걸린다 (2번 시험 학습으로 먼저 잰다).

## 준비 (그냥 실행)

In [ ]:
# 다른 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git fetch -q origin
!git checkout exp/general-json   # main 에 합쳐지기 전까지 이 브랜치
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, os, json, datetime; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
G_DIR = f"{DRIVE}/runs/G"
OURS_DIR = f"{DRIVE}/runs/full"
OFFICIAL = "data/raw/official/official_test.jsonl"
# 이름: (test 파일, 결과 폴더, ours 의 기존 결과 폴더)
SETS = {
    "dev":      ("data/test.jsonl",               f"{DRIVE}/runs/eval",          f"{DRIVE}/runs/eval/ours"),
    "official": (OFFICIAL,                        f"{DRIVE}/runs/eval_official", f"{DRIVE}/runs/eval_official/ours"),
    "style":    ("data/style/dev_styled.jsonl",   f"{DRIVE}/runs/eval/style",    f"{DRIVE}/runs/eval/style/ours"),
}
def n_lines(p): return sum(1 for _ in open(p)) if os.path.exists(p) else -1
def done_eval(name, s):
    test, base, _ = SETS[s]
    return n_lines(f"{base}/{name}/probs.jsonl") == n_lines(test) and n_lines(f"{base}/{name}/preds.jsonl") == n_lines(test)
if not os.path.exists(OFFICIAL):
    !python -m src.official_bench convert
print("G 학습 끝:", os.path.exists(f"{G_DIR}/train_args.json"))
for s in SETS: print(s, "| G 채점 끝:", done_eval("G", s), "| ours 기존 probs:", os.path.exists(f"{SETS[s][2]}/probs.jsonl"))

---
## 1. 데이터 확인
프롬프트 한 줄을 눈으로 보고, 길이를 잰다. 정답을 기존 채점기 형식으로 바꾼 것이 원래 정답과 같은지도 본다.

**예측**: G 는 우리 test 에서 ours 보다 positive 정확도가 높을까 낮을까? (함수 설명을 매번 보므로 인자 형식은 더 잘 맞출 수도)
> (여기에 예측)

In [ ]:
!python -m src.general show
!python -m src.general stats

## 2. 시험 학습 (30 step) — 메모리·속도 재기
batch 16 을 4개씩 4번에 나눠 담는다 (실제 batch 16 그대로, 계산 결과 같음). loss 는 정답 자리만 계산한다 (`AnswerOnlyLossTrainer`, 값은 기본 loss 와 같음).
확인할 것: `[데이터]` 4000개·최대 약 2,500 토큰, `[학습 전 loss]`, 최대 메모리, step 당 시간 → 본 학습 시간 = step 당 시간 × 750.
메모리가 넘치면 `--batch-size 2 --grad-accum 8`, 남으면 `--gradient-checkpointing` 을 빼 본다 (빠름).

In [ ]:
!python -m src.train --format json --batch-size 4 --grad-accum 4 --gradient-checkpointing --max-steps 30 --output-dir /content/runs/try_G

---
## 3. 본 학습 → 생성·채점·거절 점수 (dev, 공식 210, 말투 test)
이미 끝난 것은 건너뛴다. 생성(`preds`)은 기존 채점기(`src.evaluate score`)로 그대로 채점된다.

In [ ]:
if not os.path.exists(f"{G_DIR}/train_args.json"):
    print("학습 시작", datetime.datetime.now())
    !python -m src.train --format json --batch-size 4 --grad-accum 4 --gradient-checkpointing --output-dir {G_DIR}
    print("학습 끝", datetime.datetime.now())
for s, (test, base, _) in SETS.items():
    if done_eval("G", s):
        continue
    print(f"\n==================== {s}", datetime.datetime.now())
    !python -m src.general generate --model {G_DIR} --test-file {test} --out {base}/G/preds.jsonl
    !python -m src.evaluate score --test-file {test} --preds {base}/G/preds.jsonl --out {base}/G
    !python -m src.general probs --model {G_DIR} --test-file {test} --preds {base}/G/preds.jsonl --out {base}/G/probs.jsonl

---
## 4. 비교 — 핵심 표
1. 정답 채점 (greedy): 전체 정확도, positive / negative
2. 거절 지표: AUROC, dev 문턱값을 공식 210 에 옮겼을 때 positive 거절 수 (과잉 거절), 95% 구간 (ours 와 짝지은 bootstrap)
3. negative 범주별 (near-miss 인 smart_home·phone_settings·other_apps 에서 오호출이 다른가)
4. 말투별 (실험 S 와 같은 표. G 도 말투에 따라 거절이 흔들리나)

**예측**: 공식 210 의 낯선 말투 positive 를 G 는 ours(65개)보다 덜 거절할까?
> (여기에 예측)

In [ ]:
for s, (test, base, ours) in SETS.items():
    if s == "style": continue
    print(f"\n==================== {s} (greedy 채점)")
    for name, d in [("ours", ours), ("G", f"{base}/G")]:
        if os.path.exists(f"{d}/summary.json"):
            m = json.load(open(f"{d}/summary.json"))["전체"]
            print(f"  {name:<5} 전체 {m['acc']:.2%} | positive {m['acc_positive']:.2%} | negative {m['acc_negative']:.2%} | {m['outcomes']}")

In [ ]:
D = SETS["dev"]; O = SETS["official"]
print("== 우리 test (dev) 거절 지표")
!python -m src.refusal_metrics --test {D[0]} --model ours {D[2]}/probs.jsonl - --model G {D[1]}/G/probs.jsonl -
print("== 공식 210 (문턱값은 dev 에서)")
!python -m src.refusal_metrics --test {O[0]} --model ours {O[2]}/probs.jsonl {D[2]}/probs.jsonl --model G {O[1]}/G/probs.jsonl {D[1]}/G/probs.jsonl
print("== negative 범주별 (dev)")
!python -m src.negcat_report --test-file {D[0]} --model ours:irr {D[2]}/probs.jsonl --model G:irr {D[1]}/G/probs.jsonl

In [ ]:
S = SETS["style"]
!python -m src.style_report --test-file {S[0]} --model ours {S[2]}/probs.jsonl --model G {S[1]}/G/probs.jsonl --out {S[1]}/G/style_report.json | tee {S[1]}/G/style_report.txt

---
## 끝나면
결과: Drive `runs/G/` (모델), `runs/eval/G/`, `runs/eval_official/G/`, `runs/eval/style/G/` (`preds`·`scored`·`summary`·`probs`).
랩탑에서 Claude 에게 "실험 G 결과 분석해줘" 라고 하면 설계 문서 4.5절 예측과 비교한다.
MASSIVE·BFCL 채점은 모든 모델이 확정된 뒤 한 번에 한다 (설계 문서 8절). G 도 그때 같이 (`src.general generate/probs` 에 그 test 파일).